In [1]:
import numpy as np
import pandas as pd
import cv2
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import load_model

In [2]:
import cv2
import os

img = cv2.imread("../modelo_yolo/imagen_recortada.png")
alto, ancho = img.shape[:2]

cell_h = alto // 9
cell_w = ancho // 9

os.makedirs("celdas", exist_ok=True)
contador = 0

# Usamos un margen dinámico o más pequeño para evitar celdas vacías o negativas
margen_h = max(2, cell_h // 12)
margen_w = max(2, cell_w // 12)

for fila in range(9):
    for columna in range(9):
        y1 = fila * cell_h
        y2 = (fila + 1) * cell_h
        x1 = columna * cell_w
        x2 = (columna + 1) * cell_w

        celda = img[y1 + margen_h : y2 - margen_h, x1 + margen_w : x2 - margen_w]
        cv2.imwrite(f"celdas/celda_{contador}.jpg", celda)
        contador += 1

print(f"{contador} celdas guardadas con éxito.")

81 celdas guardadas con éxito.


In [3]:
import tensorflow as tf
import numpy as np

# Cargar MNIST
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = tf.keras.datasets.mnist.load_data()

# 1. Eliminar el dígito 0 original de MNIST para evitar colisiones
mask_train = y_train_raw != 0
mask_test = y_test_raw != 0

X_train = X_train_raw[mask_train]
y_train = y_train_raw[mask_train]
X_test = X_test_raw[mask_test]
y_test = y_test_raw[mask_test]

# 2. Crear imágenes completamente vacías (casilla vacía = clase 0)
imagenes_vacias = np.zeros((5000, 28, 28), dtype=np.uint8)
etiquetas_vacias = np.zeros(5000, dtype=np.uint8)

# Concatenar las nuevas imágenes vacías
X_train = np.concatenate([X_train, imagenes_vacias], axis=0)
y_train = np.concatenate([y_train, etiquetas_vacias], axis=0)

# 3. BARAJAR (Shuffle) manualmente antes del fit para corregir el problema de val_accuracy
indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

# Normalizar y redimensionar para la CNN
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    # Usamos Input(shape) según la advertencia de Keras
    tf.keras.layers.Input(shape=(28, 28, 1)),
    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax") # 10 clases (del 0 al 9)
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Ahora verás cómo la val_accuracy sube a más del 98% a la par que el entrenamiento
model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_split=0.2,
    batch_size=64
)

# ¡CRUCIAL! Guardar el modelo para poder cargarlo después
model.save("modelo2_cnn.keras")
print("Modelo guardado correctamente.")

Epoch 1/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.9457 - loss: 0.1828 - val_accuracy: 0.9810 - val_loss: 0.0610
Epoch 2/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - accuracy: 0.9841 - loss: 0.0515 - val_accuracy: 0.9865 - val_loss: 0.0433
Epoch 3/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - accuracy: 0.9881 - loss: 0.0363 - val_accuracy: 0.9866 - val_loss: 0.0405
Epoch 4/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 20s 27ms/step - accuracy: 0.9915 - loss: 0.0269 - val_accuracy: 0.9882 - val_loss: 0.0377
Epoch 5/5
739/739 ━━━━━━━━━━━━━━━━━━━━ 21s 27ms/step - accuracy: 0.9937 - loss: 0.0206 - val_accuracy: 0.9887 - val_loss: 0.0343
Modelo guardado correctamente.


In [11]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Cargamos tu modelo entrenado de clasificación
model = load_model("modelo2_cnn.keras")
tablero = []

print("Procesando las celdas del Sudoku...")

# 2. Recorremos las 81 celdas guardadas
for i in range(81):
    img_celda = cv2.imread(f"celdas/celda_{i}.jpg")

    # Forzar escala de grises de forma segura
    if len(img_celda.shape) == 3:
        img_gray = cv2.cvtColor(img_celda[:, :, :3], cv2.COLOR_BGR2GRAY)
    else:
        img_gray = img_celda.copy()

    if img_gray is None or img_gray.size == 0:
        tablero.append(0)
        continue

    # Binarización adaptativa limpia
    _, img_init_bin = cv2.threshold(
        img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    contornos_padre, _ = cv2.findContours(
        img_init_bin, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE
    )

    img_final = np.zeros((28, 28), dtype=np.uint8)
    encontrado = False
    w_box, h_box = 0, 0

    if contornos_padre:
        h_c = img_gray.shape[0]
        w_c = img_gray.shape[1]

        contornos_validos = []
        for c in contornos_padre:
            x, y, w_box_c, h_box_c = cv2.boundingRect(c)

            # --- AJUSTE DE BORDES PERMISIVO ---
            # Si el contorno toca el borde exterior pero es muy grande (un número desplazado),
            # le permitimos pasar bajando el filtro de área al 15%
            if (
                x <= 0
                or y <= 0
                or (x + w_box_c) >= w_c
                or (y + h_box_c) >= h_c
            ):
                if cv2.contourArea(c) > (w_c * h_c * 0.15):
                    contornos_validos.append(c)
                continue

            contornos_validos.append(c)

        if contornos_validos:
            # Seleccionamos el contorno con mayor área lógica
            c_numero = max(contornos_validos, key=cv2.contourArea)
            x, y, w_box, h_box = cv2.boundingRect(c_numero)

            # Filtro de ruido mínimo: permitimos números muy esbeltos o delgados (como el 1 y el 7)
            if w_box >= 2 and h_box >= 5:
                digito = img_init_bin[y : y + h_box, x : x + w_box]

                # Redimensionar conservando el aspecto MNIST
                factor = min(18 / h_box, 18 / w_box)
                nuevo_w = int(w_box * factor)
                nuevo_h = int(h_box * factor)
                digito_scaled = cv2.resize(digito, (nuevo_w, nuevo_h))

                # Centrar en lienzo negro de 28x28
                start_y = (28 - nuevo_h) // 2
                start_x = (28 - nuevo_w) // 2
                img_final[
                    start_y : start_y + nuevo_h, start_x : start_x + nuevo_w
                ] = digito_scaled
                encontrado = True

    # --- FILTRO ULTRA-PERMISIVO PARA CASILLAS VACÍAS ---
    # Bajamos el mínimo a 6 píxeles blancos activos. Si hay un 7 fino, pasará la prueba.
    if not encontrado or cv2.countNonZero(img_final) < 6:
        tablero.append(0)
        continue

    # 3. Predicción con la CNN
    img_cnn = img_final.astype("float32") / 255.0
    img_cnn = img_cnn.reshape(1, 28, 28, 1)

    pred = model.predict(img_cnn, verbose=0)
    clase_predicha = np.argmax(pred)

    # --- REGLAS DE DESEMPATE GEOMÉTRICO ---
    relacion_aspecto = w_box / float(h_box)

    # Desempate 1 vs 7 (El 7 es más ancho arriba en proporción a su alto)
    if clase_predicha == 1 and relacion_aspecto > 0.43:
        clase_predicha = 7

    # Desempate 5 vs 6 (Densidad en la mitad inferior de la celda)
    if clase_predicha == 5:
        mitad_inferior = img_final[14:, :]
        if cv2.countNonZero(mitad_inferior) > (
            cv2.countNonZero(img_final) * 0.58
        ):
            clase_predicha = 6

    tablero.append(int(clase_predicha))

# 4. Reconstrucción de la matriz 9x9
sudoku_matriz = np.array(tablero).reshape(9, 9)

# 5. IMPRESIÓN DIRECTA EN PANTALLA
print("\n=====================================")
print("     MATRIZ DEL SUDOKU DETECTADA     ")
print("=====================================")
print(sudoku_matriz)
print("=====================================")

Procesando las celdas del Sudoku...

     MATRIZ DEL SUDOKU DETECTADA     
[[0 0 2 7 0 6 4 9 3]
 [5 0 0 0 0 0 6 0 0]
 [0 0 7 0 2 0 5 8 0]
 [0 0 5 6 3 0 8 0 0]
 [0 0 3 0 0 5 0 0 2]
 [4 7 8 0 0 9 0 0 0]
 [0 5 0 0 0 0 2 0 6]
 [3 0 0 0 4 0 0 0 0]
 [0 0 0 3 6 0 7 0 0]]


In [12]:
# --- GUARDAR LA MATRIZ PARA EL MODELO 3 ---
# Opción A: Guardarlo en formato binario de Numpy (.npy) -> Es la más limpia
np.save("sudoku_detectado.npy", sudoku_matriz)

# Opción B (Opcional): Guardarlo como un archivo de texto común (.txt) por si quieres abrirlo a mano y verlo
np.savetxt("sudoku_detectado.txt", sudoku_matriz, fmt="%d")

print("¡Matriz exportada con éxito para el Modelo 3!")

¡Matriz exportada con éxito para el Modelo 3!
